# MPM 204: Assignment 3

### Due 23th April 2026 9AM

#### For all questions, you must show your work. This will enable us to understand your thought process, give partial credit, and prevent crude cheating.
#### Please make sure that you are not simply copying each other's code, but rather striving to understand each other's work and learn from it.
#### Additionally, please provide the R code at the end of your solution and include R commands along with R outputs. This will help to describe your solutions more clearly.

In [ ]:
## load required packages

* The data set was developed by Leathwick et al. in 2008 for research and conservation planning in New Zealand. It describes species data, which were recorded at 13,369 sites across New Zealand's rivers, spanning the major environmental gradients.
* *Anguilla australis* was caught at 20% of these sites. This data set is much larger than what is typically available in ecology. Therefore, we subsample the sites, partitioning off some records for modeling and keeping the rest for independent evaluation.  

* The explanatory variables used in this study were a set of 11 functionally relevant environmental predictors (Table 1).
* These variables summarize conditions over several spatial scales, including the local (segment and reach) scale, upstream catchment scale, and downstream to the sea.
* Most of these variables were available as GIS data for the full river system in New Zealand, enabling us to make predictions for all rivers.
* However, one variable, `LocSed`, which described local substrate conditions, only had records at 82% of the sites. The 12th variable was categorical, describing the fishing method (Table 1).

## Aim of the analysis
### Given these records and covariates, we want to model the joint probability of occurrence and capture of *A. australis*.

## Section 0: Model data

#### Q1: Read the data file "A_australis_model_data.csv" in R and answer the following subquestions
Q1a: Number of rows and columns  
Q1b: Number of possible covariates in the data  
Q1c: Identify number of quantitative and categorical variables in the data   
Q1d: For each possible covariate, how many missing values are present?

#### Points: 5

In [ ]:
data<-
head(data)

### Understanding the dataset

The following is a list of variables and their descriptions.

| Covariate | Description | Mean and range |
|-----------|-------------|----------------|
| **Reach scale covariate** |  |  |
| LocSed | Weighted average of proportional cover of bed sediment | 3.77 (1–7) |
| **Segment scale covariates** |  |  |
| SegSumT | Summer air temperature (°C) | 16.3 (8.9–19.8) |
| SegTSeas | Winter air temperature (°C), normalized with respect to SegJanT | 0.36 (–4.2–4.1) |
| SegLowFlow | Segment low flow (m³ s⁻¹), fourth root transformed | 1.092 (1.0–4.09) |
| **Downstream covariates** |  |  |
| DSDist | Distance to coast (km) | 74 (0.03–433.4) |
| DSDam | Presence of known downstream obstructions, mostly dams | 0.18 (0 or 1) |
| DSMaxSlope | Maximum downstream slope (°) | 3.1 (0–29.7) |
| **Upstream/catchment scale covariates** |  |  |
| USAvgT | Average temperature in catchment (°C) compared with segment, normalized with respect to SegJanT | –0.38 (–7.7–2.2) |
| USRainDays | Days per month with rain >25 mm | 1.22 (0.21–3.30) |
| USSlope | Average slope in the upstream catchment (°) | 14.3 (0–41.0) |
| USNative | Area with indigenous forest (proportion) | 0.57 (0–1) |
| **Fishing method** |  |  |
| Method | Fishing method in five classes: electric, net, spot, trap, mixture | NA |

`LocSed`: 1 = mud, 2 = sand, 3 = fine gravel; 4 = coarse gravel; 5 = cobble; 6 = boulder; 7 = bedrock    
Even though the interpretation for `LocSed` can be categorical, this is a weighted average of proportion of cover of bed sediment. Researchers have converted their readings into numeric data; therefore, it is not necessary to convert this variable into a factor.

## Section 1: Data exploration

Q2. Find the quantitative variables that are highly correlated with each other.  
hint: use correlation plots shown previously in class.  
Points: 2  

In [ ]:
head(data[])

In [ ]:
plot(data[])

* The plot function can be difficult to read with large data points.
* Let's use another plotting function from `ggplot` to explore the correlation between our covariates.

In [ ]:
covariates_numeric = names(data)[!(names(data) %in% c('Site', 'Angaus', "Method"))]
covariates_numeric

In [ ]:
cormat <- round(cor(data[,covariates_numeric]),2)
cormat

#### since there are missing values in *LocSed* lets redo the correlation matrix by dropping rows with missing data

In [ ]:
cormat <- round(cor(drop_na(data[,])),2)
cormat

#### Q3: Which two covariates have the highest correlation coefficient? Are there any other pairs that concern you?
Q3a: Which correlation coefficient is being calculated by the command cor? What is the default method for calculating the correlation coefficient? What other methods are possible?  
Points: 5  

#### Plotting correlation matrix for easier visualization

#### Q4: Use following code to plot the correlation matrix you generated above  
Points: 2

In [ ]:
library(reshape2)
melted_cormat <- melt(cormat)
head(melted_cormat)


In [ ]:
#### complete the command below to get the desired plot
ggplot(data = , aes(x=Var1, y=Var2, fill=value)) +
  geom_tile()

#### Q5: Let's make the plot much more readable. Use the following code to create plots for your correlation matrix.
Change the title of the legend bar for the plot based on the correlation coefficient you are calculating.  
Points: 3

In [ ]:
### function to reorder the correlation matrix
reorder_cormat <- function(cormat){
# Use correlation between variables as distance
dd <- as.dist((1-cormat)/2)
hc <- hclust(dd)
cormat <-cormat[hc$order, hc$order]
}

#### Which dataframe object will you use here in the command below?

In [ ]:
# Reorder the correlation matrix
#### complete the command below to get the desired output
cormat <- reorder_cormat()

In [ ]:
# Get upper triangle of the correlation matrix
get_upper_tri <- function(cormat){
cormat[lower.tri(cormat)]<- NA
return(cormat)
}

## remove the upper half of the matrix
upper_tri <- get_upper_tri(cormat)

In [ ]:
melted_cormat <- melt(upper_tri, na.rm = TRUE)
# Create a ggheatmap
ggheatmap <- ggplot(melted_cormat, aes(Var2, Var1, fill = value))+
 geom_tile(color = "white")+
 scale_fill_gradient2(low = "blue", high = "red", mid = "white",
   midpoint = 0, limit = c(-1,1), space = "Lab",
    name="") +
  theme_minimal()+ # minimal theme
 theme(axis.text.x = element_text(angle = 45, vjust = 1,
    size = 12, hjust = 1))+
 coord_fixed()
# Print the heatmap
ggheatmap +
geom_text(aes(Var2, Var1, label = value), color = "black", size = 4) +
theme(
  axis.title.x = element_blank(),
  axis.title.y = element_blank(),
  panel.grid.major = element_blank(),
  panel.border = element_blank(),
  panel.background = element_blank(),
  axis.ticks = element_blank(),
  legend.justification = c(1, 0),
  legend.position = c(0.6, 0.7),
  legend.direction = "horizontal")+
  guides(fill = guide_colorbar(barwidth = 7, barheight = 1,
                title.position = "top", title.hjust = 0.5))

#### I will keep a close eye on covariates that have correlation coefficient higher than 0.2%. (more than 20% correlation)

## Section 2: Univariate model building and exploration

##### Q6: Use the following code to run univariate models.
Q6a: List the significant covariates.   
Q6b: Scale the covariates and rerun the analysis.  
Q6c: What differences do you see between the scaled and unscaled (raw) analyses? Did any new covariates become significant?  

Points: 6

In [ ]:
#### complete the command below to get the desired plot and output

library(plyr)
covariates <- names(data)[!( names(data) %in% c('Site', 'Angaus', "Method"))]
getBetaAndSe <- function(covar){
    coefs <- glm() %>%
        summary %>% coef
    data.frame(variable = covar,
               estimate = coefs[2,1],
               se = coefs[2,2],
               p = coefs[2,4])
}

require(ggthemes)
beta.table <- adply(t(covariates), 2, getBetaAndSe) %>%
            arrange(estimate) %>%
            mutate(low = estimate - 2*se,
                         high = estimate + 2*se,
                         variable = reorder(variable, order(estimate)))

# note - the "reorder" was necessary to make the variables plot
# in the right order in ggplot ... but it took TOO LONG to figure out!

ggplot(beta.table, aes(y=estimate, x=variable, ymin=low, ymax=high, color = (p < 0.05))) +
            geom_pointrange() +
            geom_hline(yintercept = 0, col="darkgrey", lty = 3, lwd=2) +
            coord_flip() + theme_few() + ggtitle("Single main effect plot")

In [ ]:
#### complete the command below to get the desired plot and output
covariates <- names(data)[!( names(data) %in% c('Site', 'Angaus', "Method"))]
getBetaAndSe <- function(covar){
    coefs <- glm() %>%
        summary %>% coef
    data.frame(variable = covar,
               estimate = coefs[2,1],
               se = coefs[2,2],
               p = coefs[2,4])
}

require(ggthemes)
beta.table <- adply(t(covariates), 2, getBetaAndSe) %>%
            arrange(estimate) %>%
            mutate(low = estimate - 2*se,
                         high = estimate + 2*se,
                         variable = reorder(variable, order(estimate)))

# note - the "reorder" was necessary to make the variables plot
# in the right order in ggplot ... but it took TOO LONG to figure out!

ggplot(beta.table, aes(y=estimate, x=variable, ymin=low, ymax=high, color = (p < 0.05))) +
            geom_pointrange() +
            geom_hline(yintercept = 0, col="darkgrey", lty = 3, lwd=2) +
            coord_flip() + theme_few() + ggtitle("Single main effect plot")

#### Q7: Run a single univariate model for categorical covariate

Points: 2

In [ ]:
cat_model =
summary(cat_model)

## Section 3: Model selection
#### Q8: Model selection procedure

This is the main section of the assignment.
Please closely follow the instructions below.  
###### Start by creating a global model.
* Start with a global model. Covariates for global model will be decided on
1. Correlation coefficient: Select one independent variable from the covariates that have an $r^2>0.7$. In this dataset, there are a couple of covariate pairs that have an $r^2$ close to 0.7. Select the most relevant covariate.
2. Do not include covariates that are insignificant in the univariate analysis.
3. Explore the global model and inform which covariates are significant. Run the model by removing rows with missing values.
4. Use the `anova` function to check if the inclusion of covariates significantly improves the model fit (by increasing the likelihood).
5. Describe the results of the `anova` test (analysis of deviance).
6. Use the `stepAIC` function (in `MASS`) to march through the inclusion and removal of different covariates to find the one with the lowest AIC.
7. The outputs of `stepAIC` will not result in multiple steps for this dataset. The model is already close to MAM.
8. Remove covariates from the model resulted from the `stepAIC` function and re-fit the model with reduced covariates. Check AIC for it. What do you observe in terms of $B$ estimates, $odds$ $ratios$, and $significance$?

#### Points 25

In [ ]:
## add Covariates that you want to exclude from the main model in the list below along with Site and Angaus (output variable)
covariates_main_effects <- names(data)[!( names(data) %in% c('Site', 'Angaus'))]
covariates_main_effects

#### Main effects model

In [ ]:
## remove missing data from our dataframe. Use the this data for your model building.
data_red = drop_na(data)

In [ ]:
maineffects.glm <-
summary(maineffects.glm)

In [ ]:
print(anova(maineffects.glm, )) ## complete the code to get desired outputs

In [ ]:
require(MASS)
print(best.main.glm <- stepAIC(), trace = 0) ## complete the code to get desired outputs

In [ ]:
## Removing non-signifacnt covariates from the model and reruning the model
maineffects.glm2 <-
summary(maineffects.glm2)

In [ ]:
#### compare two models for their measures of effects
tab_model(maineffects.glm, maineffects.glm2, show.reflvl = TRUE)

## Section 4: Interactions
#### Q9: Selection of interactions

1. Create a dataframe including the output column and all covariates to be tested in the interaction model, removing missing data based on the results of the previous model.
2. Run a full interaction model using the code provided below.
3. Use the `stepAIC` function to find the model with the lowest AIC.
4. Conduct an `ANOVA` test on the best-fitting (lowest AIC) model to identify terms that significantly improve the likelihood.
5. List the interactions that significantly improve the likelihood of the model.
6. Create a final model that includes both interaction terms and main effects.
7. Compare this model to the main effects model using AIC values and odds ratios.

#### Points: 25  

In [ ]:
### Create a dataframe with output column and all covariates that you want to test in the interactions model (remove missing data)
interaction_data = data_red[, c("Angaus", COVARIATES)]## complete the code to get desired outputs

In [ ]:
interaction.glm <- glm(Angaus ~ (.)^2,data = interaction_data, family = "binomial")
best.interaction.glm <- stepAIC(, trace = 0)## complete the code to get desired outputs

In [ ]:
summary(best.interaction.glm)

In [ ]:
## complete the code to get desired outputs
print(anova(, test = "Chi"))

In [ ]:
## Run a final model with significant main effects and signficant interactions

In [ ]:
final_model =
summary(final_model)

In [ ]:
## Compare model that has interaction terms with model that does not have interaction terms for its measures of effects and AIC
tab_model(MODEL1, MODEL2 show.reflvl = TRUE)

In [ ]:
AIC(MODEL1, MODEL2)

## Secton 5: Summarize and present your methods and results
1. Briefly describe, in bullet points, all the steps you took for model selection, starting from data exploration to finalizing the model. Importantly, tell us which results led you to make a specific decision (dropping or selecting a covariate/model).
2. Present your final model result. Describe in words the model results (odds) for two significant main effects and two interactions (if you have any).

### Points: 20